# Capítulo 4 · Laboratório 4: tendência, suportes e gaps
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Gaps que fecham e que não fecham, scanner de exemplos, suportes e resistências com contagem de toques, linha de tendência interna automática, retrações, dias de reversão, filtros de rompimento, números redondos e o Forcado de Andrews. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap04.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap04.pkl"
pc = lambda x, k=0: f"{x:.{k}%}".replace(".", ",")
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ACOES = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "B3SA3", "ABEV3", "WEGE3", "RENT3", "SUZB3",
         "GGBR4", "ITSA4", "AXIA3", "PRIO3", "RADL3", "LREN3", "BPAC11", "SBSP3", "EQTL3", "VIVT3"]
INICIO = {"^BVSP": "2000-01-01", "^DJI": "1998-01-01"}


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=False, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for t in [a + ".SA" for a in ACOES] + list(INICIO):
    if t not in D:
        D[t] = baixa(t, INICIO.get(t, "2015-01-01"))
confere({t: D[t] for t in INICIO})
pd.to_pickle(D, CACHE)

## Parte 1: dados e preço ajustado

In [ ]:
def limpa(d, indice=False):
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])
    if not indice:
        ok &= d["Volume"] > 0
    return d[ok].copy()


def ajusta(d):
    """Multiplica abertura, máxima, mínima e fechamento pelo fator do fechamento ajustado.
    Sem isso, a queda da data ex-dividendo aparece como um gap que não existiu."""
    f = d["Adj Close"] / d["Close"]
    a = d.copy()
    for c in ("Open", "High", "Low", "Close"):
        a[c] = d[c] * f
    return a


B = {a: limpa(D[a + ".SA"]) for a in ACOES}          # preço de tela (para os gráficos)
A = {a: ajusta(B[a]) for a in ACOES}                  # preço ajustado (para a estatística)
IBOV, DJI = limpa(D["^BVSP"], True), limpa(D["^DJI"], True)
print("Parte 1. Pregões por ação depois da limpeza:", {a: len(B[a]) for a in ACOES[:5]}, "...")

## Parte 2: gaps fecham sempre?

In [ ]:
def gaps(d, minimo=0.0):
    """Gap de abertura: abre acima da máxima anterior (alta) ou abaixo da mínima anterior (baixa).
    O gap 'fecha' quando o preço volta a tocar a máxima (ou a mínima) do dia anterior.
    dias = 0: fechou no próprio pregão; NaN: não fechou até 30/09/2026."""
    o, h, l = d["Open"].to_numpy(), d["High"].to_numpy(), d["Low"].to_numpy()
    lin = []
    for i in range(1, len(d)):
        if o[i] > h[i - 1] * (1 + minimo):
            tipo, alvo = "alta", h[i - 1]
            toque = l[i:] <= alvo
        elif o[i] < l[i - 1] * (1 - minimo):
            tipo, alvo = "baixa", l[i - 1]
            toque = h[i:] >= alvo
        else:
            continue
        dias = int(toque.argmax()) if toque.any() else np.nan
        lin.append({"data": d.index[i], "i": i, "tipo": tipo, "alvo": alvo, "tam": o[i] / alvo - 1, "dias": dias})
    return pd.DataFrame(lin)


def resumo(g):
    r = {"gaps": len(g)}
    for p, nome in ((0, "no pregão"), (5, "até 5"), (20, "até 20"), (60, "até 60")):
        r[nome] = (g["dias"] <= p).mean()
    r["aberto"] = g["dias"].isna().mean()
    return r


G = {a: gaps(A[a]) for a in ACOES}
TAB = pd.DataFrame({a: resumo(G[a]) for a in ACOES}).T
TODOS = pd.concat(G.values())
GRANDES = TODOS[TODOS["tam"].abs() >= 0.02]
TAB.loc["Todas (20 ações)"] = resumo(TODOS)
TAB.loc["Só gaps de 2% ou mais"] = resumo(GRANDES)
print("\nParte 2. Gaps de abertura desde 2015 (preço ajustado). Fração que fechou:")
print(TAB.assign(gaps=TAB["gaps"].astype(int)).to_string(formatters={c: pc for c in TAB.columns if c != "gaps"}))
SEM_AJUSTE = pd.concat([gaps(B[a]) for a in ACOES])
print(f"   Sem ajuste de proventos: {len(SEM_AJUSTE)} gaps (contra {len(TODOS)}); a diferença são datas ex-dividendo.")

In [ ]:
import matplotlib.pyplot as plt
cols = ["no pregão", "até 5", "até 20", "até 60"]
ax = TAB.loc[["Todas (20 ações)", "Só gaps de 2% ou mais"], cols].astype(float).T.plot.bar(
    figsize=(8, 3.5), color=["white", "dimgray"], edgecolor="black", rot=0)
ax.set_ylabel("gaps que fecharam"); ax.set_title("Gaps fecham sempre? 20 ações da B3, 2015 a 2026")
plt.show()

## Parte 3: gaps que não fecharam (Deemer)

In [ ]:
def janelas(d, minimo=0.02):
    """Janela (gap que aparece no gráfico): o dia inteiro fica acima da máxima anterior (ou abaixo da mínima)."""
    h, l = d["High"].to_numpy(), d["Low"].to_numpy()
    lin = []
    for i in range(1, len(d)):
        if l[i] > h[i - 1] * (1 + minimo):
            tipo, borda = "alta", h[i - 1]
            toque = l[i:] <= borda
        elif h[i] < l[i - 1] * (1 - minimo):
            tipo, borda = "baixa", l[i - 1]
            toque = h[i:] >= borda
        else:
            continue
        dias = int(toque.argmax()) if toque.any() else np.nan
        lin.append({"data": d.index[i], "i": i, "tipo": tipo, "borda": borda, "outra": l[i] if tipo == "alta" else h[i],
                    "tam": abs(l[i] / h[i - 1] - 1) if tipo == "alta" else abs(h[i] / l[i - 1] - 1), "dias": dias})
    return pd.DataFrame(lin)


ABERTOS = []
for a in ACOES:
    j = janelas(A[a])
    j = j[j["dias"].isna() & (j["data"] <= "2025-09-30")]          # aberto há pelo menos um ano
    for _, r in j.iterrows():
        dist = A[a]["Close"].iloc[-1] / r["borda"] - 1
        ABERTOS.append({"ativo": a, **r.to_dict(), "distancia_hoje": dist})
ABERTOS = pd.DataFrame(ABERTOS).sort_values("data")
print(f"\nParte 3. Janelas de 2% ou mais abertas há mais de um ano e nunca fechadas: {len(ABERTOS)}")
print(ABERTOS[["ativo", "data", "tipo", "tam", "distancia_hoje"]].tail(12).to_string(
    index=False, formatters={"tam": pc, "distancia_hoje": pc, "data": lambda x: x.strftime("%d/%m/%Y")}))

## Parte 4: scanner de exemplos

In [ ]:
def atr(d, n=14):
    tr = pd.concat([d["High"] - d["Low"], (d["High"] - d["Close"].shift()).abs(),
                    (d["Low"] - d["Close"].shift()).abs()], axis=1).max(axis=1)
    return tr.rolling(n).mean()


def scanner(d, desde="2023-01-01"):
    """Classifica as janelas de alta pela vizinhança, como no texto:
    quebra   = depois de 60 pregões sem alta, salta acima da máxima de 20 pregões, com volume, e não fecha em 20;
    figura   = sai de um retângulo com pelo menos 3 toques no teto (os 3 toques de Aksel Kibar);
    fuga     = no meio de uma alta já em andamento, sem fechar em 20 pregões;
    exaustão = depois de alta forte (25% em 40 pregões), com volume alto, fecha em até 5 pregões e o preço cai;
    ilha     = janela de alta seguida, em até 15 pregões, de janela de baixa (ilha de topo)."""
    h, l, c, v = (d[k].to_numpy() for k in ("High", "Low", "Close", "Volume"))
    vm = d["Volume"].rolling(20).mean().to_numpy()
    out = []
    j = janelas(d, 0.01)
    for _, r in j[(j["tipo"] == "alta") & (j["data"] >= desde)].iterrows():
        i = r["i"]
        if i < 60 or i > len(d) - 21:
            continue
        base_h, base_l = h[i - 30:i].max(), l[i - 30:i].min()
        alta40 = c[i - 1] / l[i - 40:i].min() - 1
        volx = v[i] / vm[i - 1] if vm[i - 1] > 0 else np.nan
        fechou20 = not np.isnan(r["dias"]) and r["dias"] <= 20
        depois = c[min(i + 20, len(d) - 1)] / c[i] - 1
        if base_h / base_l - 1 < 0.12 and l[i] > base_h and volx > 1.5 and not fechou20:
            toques = int(((h[i - 60:i] >= base_h * 0.985) & (np.r_[0, np.diff(h[i - 60:i])] > 0)).sum())
            if toques >= 3:
                out.append({"tipo": "figura", "data": r["data"], "tam": r["tam"], "vol_x": volx,
                            "toques_teto": toques, "depois20": depois})
        if c[i - 1] < c[i - 60] * 1.03 and l[i] > h[i - 20:i].max() and volx > 1.5 and not fechou20:
            out.append({"tipo": "quebra", "data": r["data"], "tam": r["tam"], "vol_x": volx, "depois20": depois})
        if 0.10 < alta40 < 0.40 and not fechou20 and c[i - 1] > base_h * 0.97:
            antes = c[i - 1] / l[i - 40:i].min() - 1
            ext = h[i:i + 40].max() / c[i - 1] - 1
            out.append({"tipo": "fuga", "data": r["data"], "tam": r["tam"], "vol_x": volx,
                        "razao_medida": ext / antes, "depois20": depois})
        if alta40 > 0.25 and volx > 1.8 and not np.isnan(r["dias"]) and r["dias"] <= 5 and depois < -0.05:
            out.append({"tipo": "exaustao", "data": r["data"], "tam": r["tam"], "vol_x": volx, "depois20": depois})
        for k in range(2, 16):
            if i + k >= len(d):
                break
            topo_ilha_baixo = l[i:i + k].min()
            if h[i + k] < topo_ilha_baixo and topo_ilha_baixo > h[i - 1]:
                out.append({"tipo": "ilha", "data": r["data"], "fim": d.index[i + k], "tam": r["tam"], "dias_ilha": k,
                            "depois20": c[min(i + k + 20, len(d) - 1)] / c[i + k] - 1})
                break
            if l[i + k] <= h[i - 1]:
                break
    return pd.DataFrame(out)


SCAN = pd.concat([scanner(B[a]).assign(ativo=a) for a in ACOES], ignore_index=True)
print("\nParte 4. Exemplos encontrados desde 2023 (preço de tela):", SCAN["tipo"].value_counts().to_dict())
for tp in ("quebra", "figura", "fuga", "exaustao", "ilha"):
    s = SCAN[SCAN["tipo"] == tp].sort_values("data").tail(4)
    print(f"   {tp}:", "; ".join(f"{r.ativo} {r.data:%d/%m/%Y} ({pc(r.tam, 1)}, depois {pc(r.depois20, 1)})"
                                  for r in s.itertuples()))

## Parte 5: suportes e resistências automáticos

In [ ]:
def pivos(d, janela=5):
    alto, baixo = d["High"], d["Low"]
    topos = alto[alto == alto.rolling(2 * janela + 1, center=True).max()]
    fundos = baixo[baixo == baixo.rolling(2 * janela + 1, center=True).min()]
    return topos, fundos


def niveis(d, janela=5, tolerancia=0.015, minimo_toques=3):
    """Agrupa topos e fundos próximos (tolerância em %) e conta os toques de cada nível."""
    topos, fundos = pivos(d, janela)
    p = np.sort(np.r_[topos.to_numpy(), fundos.to_numpy()])
    grupos, atual = [], [p[0]]
    for x in p[1:]:
        if x <= atual[0] * (1 + tolerancia):
            atual.append(x)
        else:
            grupos.append(atual)
            atual = [x]
    grupos.append(atual)
    t = pd.DataFrame({"nivel": [np.mean(g) for g in grupos], "toques": [len(g) for g in grupos]})
    return t[t["toques"] >= minimo_toques].sort_values("toques", ascending=False)


PETR = B["PETR4"].loc["2025-10-01":]
NIV = niveis(PETR, janela=3, tolerancia=0.012)
print(f"\nParte 5. PETR4 (outubro de 2025 a setembro de 2026): níveis com 3 ou mais toques")
print(NIV.head(6).to_string(index=False, formatters={"nivel": nr}))
print("   Último fechamento:", nr(PETR["Close"].iloc[-1]))

## Parte 6: linha de tendência interna automática

In [ ]:
def linha_interna(d, janela=3, tolerancia=0.008, min_dist=20, max_mes=0.04, cobertura=0.5):
    """Testa as retas que passam por dois pivôs quaisquer (topos ou fundos) e fica com a que
    encosta em mais pivôs. É o que o analista faz no olho, só que testando todas as combinações.
    Filtros: inclinação de no máximo max_mes por mês (linha muito íngreme não se sustenta) e toques
    espalhados por pelo menos 'cobertura' do período."""
    topos, fundos = pivos(d, janela)
    pts = pd.concat([topos, fundos]).sort_index()
    x = np.array([d.index.get_loc(k) for k in pts.index])
    y = np.log(pts.to_numpy())
    melhor = (0, None)
    for a in range(len(x)):
        for b in range(a + 1, len(x)):
            if x[b] - x[a] < min_dist:
                continue
            m = (y[b] - y[a]) / (x[b] - x[a])
            if abs(np.expm1(m * 21)) > max_mes:
                continue
            dist = np.abs(y - (y[a] + m * (x - x[a])))
            perto = x[dist < tolerancia]
            if perto.max() - perto.min() < cobertura * len(d):
                continue
            toques = int((dist < tolerancia).sum())
            if toques > melhor[0]:
                melhor = (toques, (x[a], y[a], m, x[dist < tolerancia]))
    return melhor


TOQ, (X0, Y0, M, XT) = linha_interna(PETR)
print(f"\nParte 6. PETR4: a melhor linha interna encosta em {TOQ} pivôs; inclinação de {pc(np.expm1(M * 21), 1)} por mês.")

In [ ]:
import numpy as np
xs = np.arange(len(PETR))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(xs, PETR["Close"], color="black", lw=0.8)
ax.plot(xs, np.exp(Y0 + M * (xs - X0)), color="gray", lw=1.2)
for _, r in NIV.head(3).iterrows():
    ax.axhline(r["nivel"], color="gray", ls=":", lw=0.8)
ax.set_ylim(PETR["Low"].min() * 0.95, PETR["High"].max() * 1.05)
ax.set_title(f"PETR4: linha interna com {TOQ} toques e níveis com mais toques"); plt.show()

## Parte 7: retrações de 33%, 50% e 67%

In [ ]:
def zigzag(s, limite=0.10):
    """Pivôs de um ziguezague: a perna só muda quando o preço volta mais que o limite."""
    v = s.to_numpy()
    piv, tend, imax, imin = [], 0, 0, 0
    for i in range(1, len(v)):
        if tend >= 0 and v[i] > v[imax]:
            imax = i
        if tend <= 0 and v[i] < v[imin]:
            imin = i
        if tend >= 0 and v[i] < v[imax] * (1 - limite) and imax > (piv[-1][0] if piv else -1):
            piv.append((imax, v[imax], "topo")); tend, imin = -1, i
        elif tend <= 0 and v[i] > v[imin] * (1 + limite) and imin > (piv[-1][0] if piv else -1):
            piv.append((imin, v[imin], "fundo")); tend, imax = 1, i
    return piv


IB = IBOV.loc["2025-06-01":]
ZZ = zigzag(IB["Close"], 0.06)
print("\nParte 7. Ibovespa: pivôs do ziguezague de 6% (fechamento):")
for i, p, k in ZZ[-6:]:
    print(f"   {k:5s} {IB.index[i]:%d/%m/%Y} {nr(p, 0)}")
F1 = max((p for p in ZZ if p[2] == "topo"), key=lambda p: p[1])
F0 = min((p for p in ZZ if p[2] == "fundo" and p[0] < F1[0]), key=lambda p: p[1])
for f in (1 / 3, 0.5, 2 / 3, 0.382, 0.618):
    print(f"   retração de {pc(f, 1)}: {nr(F1[1] - f * (F1[1] - F0[1]), 0)}")
print(f"   menor fechamento depois do topo: {nr(IB['Close'].iloc[F1[0]:].min(), 0)}"
      f" ({pc((F1[1] - IB['Close'].iloc[F1[0]:].min()) / (F1[1] - F0[1]), 1)} da alta)")
F00 = IBOV.loc["2025-01-01":"2025-05-31", "Close"]
print(f"   medindo a alta desde o fundo de {F00.idxmin():%d/%m/%Y} ({nr(F00.min(), 0)}), a mesma queda vira"
      f" {pc((F1[1] - IB['Close'].iloc[F1[0]:].min()) / (F1[1] - F00.min()), 1)}: o resultado depende de onde se começa a medir")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(len(IB)), IB["Close"], color="black")
for f in (1 / 3, 0.5, 2 / 3):
    ax.axhline(F1[1] - f * (F1[1] - F0[1]), color="gray", ls="--", lw=0.8)
ax.set_title("Ibovespa: retrações de 33%, 50% e 67% da alta"); plt.show()

## Parte 8: dias de reversão

In [ ]:
def dias_reversao(d, n=20):
    """Topo: faz máxima de n pregões e fecha abaixo do fechamento anterior, com volume acima da média.
    Fundo: faz mínima de n pregões e fecha acima do fechamento anterior, com volume acima da média."""
    vm = d["Volume"].rolling(20).mean()
    topo = (d["High"] >= d["High"].rolling(n).max()) & (d["Close"] < d["Close"].shift()) & (d["Volume"] > vm)
    fundo = (d["Low"] <= d["Low"].rolling(n).min()) & (d["Close"] > d["Close"].shift()) & (d["Volume"] > vm)
    r = pd.DataFrame({"topo": topo, "fundo": fundo})
    r["ret10"] = d["Close"].shift(-10) / d["Close"] - 1
    return r


def estat_reversao(d):
    r = dias_reversao(d).dropna()
    return (r.loc[r["topo"], "ret10"] < 0).mean(), (r.loc[r["fundo"], "ret10"] > 0).mean(), \
        (r["ret10"] < 0).mean(), (r["ret10"] > 0).mean(), r["topo"].sum(), r["fundo"].sum()


EST = np.array([estat_reversao(A[a]) for a in ACOES])
print("\nParte 8. Dias de reversão nas 20 ações (2015 a 2026, preço ajustado):")
print(f"   topos: {int(EST[:, 4].sum())} casos; caiu em 10 pregões em {pc(np.average(EST[:, 0], weights=EST[:, 4]))}"
      f" (um dia qualquer: {pc(EST[:, 2].mean())})")
print(f"   fundos: {int(EST[:, 5].sum())} casos; subiu em 10 pregões em {pc(np.average(EST[:, 1], weights=EST[:, 5]))}"
      f" (um dia qualquer: {pc(EST[:, 3].mean())})")

## Parte 9: filtros de rompimento

In [ ]:
def filtros(d, n=20, olhar=10):
    """Rompimento da máxima de n pregões (pelo fechamento) e cinco jeitos de confirmar.
    'Falso' = volta a fechar abaixo do nível rompido em até 'olhar' pregões depois do sinal.
    'Custo' = quanto acima do nível o sinal manda entrar (em %)."""
    c, h = d["Close"], d["High"]
    nivel = h.rolling(n).max().shift()
    a14 = atr(d)
    semana = c.resample("W-FRI").last()
    sem_ok = (semana > nivel.resample("W-FRI").last()).reindex(c.index, method="ffill").fillna(False)
    regras = {"Fechamento acima": c > nivel,
              "Filtro de 3%": c > nivel * 1.03,
              "Dois dias acima": (c > nivel) & (c.shift() > nivel.shift()),
              "Fechamento da semana": sem_ok & (c.index.dayofweek == 4) & (c > nivel),
              "Meio ATR acima": c > nivel + 0.5 * a14}
    lin = {}
    for nome, sinal in regras.items():
        novo = sinal & ~sinal.shift(fill_value=False)
        idx = np.where(novo.to_numpy())[0]
        idx = idx[idx < len(d) - olhar]
        falso = [(c.iloc[i + 1:i + 1 + olhar] < nivel.iloc[i]).any() for i in idx]
        custo = [(c.iloc[i] / nivel.iloc[i] - 1) for i in idx]
        lin[nome] = (len(idx), np.mean(falso), np.mean(custo))
    return lin


FIL = {}
for a in ACOES:
    for k, (n, f, cu) in filtros(A[a]).items():
        FIL.setdefault(k, []).append((n, f, cu))
QF = pd.DataFrame({k: {"sinais": sum(x[0] for x in v), "falsos": np.average([x[1] for x in v], weights=[x[0] for x in v]),
                       "custo": np.average([x[2] for x in v], weights=[x[0] for x in v])} for k, v in FIL.items()}).T
print("\nParte 9. Rompimento da máxima de 20 pregões nas 20 ações (2015 a 2026):")
print(QF.to_string(formatters={"sinais": lambda x: f"{int(x)}", "falsos": pc, "custo": lambda x: pc(x, 1)}))

## Parte 10: o muro dos números redondos

In [ ]:
def primeiro_acima(s, nivel):
    acima = s[s >= nivel]
    return (acima.index[0], acima.iloc[0]) if len(acima) else (None, None)


ib = IBOV.loc["2025-01-01":]
print("\nParte 10. Números redondos:")
d1, v1 = primeiro_acima(IBOV["Close"], 100_000)
print(f"   Ibovespa: primeiro fechamento acima de 100 mil em {d1:%d/%m/%Y} ({nr(v1, 0)})")
print(f"   Ibovespa: máxima de fechamento {nr(ib['Close'].max(), 0)} em {ib['Close'].idxmax():%d/%m/%Y};"
      f" máxima do pregão {nr(ib['High'].max(), 0)} em {ib['High'].idxmax():%d/%m/%Y};"
      f" pregões com máxima acima de 195 mil: {(ib['High'] >= 195_000).sum()};"
      f" fechamentos acima de 200 mil: {(ib['Close'] >= 200_000).sum()}")
for nivel in (10_000, 50_000):
    dd, vv = primeiro_acima(DJI["Close"], nivel)
    print(f"   Dow Jones: primeiro fechamento acima de {nr(nivel, 0)} em {dd:%d/%m/%Y} ({nr(vv, 2)})")

## Parte 11: Forcado de Andrews

In [ ]:
def forcado(d, p1, p2, p3):
    """Três pivôs (índice, preço). A linha mediana sai de p1 e passa pelo meio de p2 e p3;
    as outras duas linhas são paralelas a ela, passando por p2 e por p3."""
    xm, ym = (p2[0] + p3[0]) / 2, (p2[1] + p3[1]) / 2
    m = (ym - p1[1]) / (xm - p1[0])
    x = np.arange(p1[0], len(d))
    return x, p1[1] + m * (x - p1[0]), p2[1] + m * (x - p2[0]), p3[1] + m * (x - p3[0])


IBF = IBOV.loc["2025-10-01":]
ZF = zigzag(IBF["Close"], 0.04)
PV = [(i, IBF["Low"].iloc[max(i - 2, 0):i + 3].min() if k == "fundo" else IBF["High"].iloc[max(i - 2, 0):i + 3].max(), k)
      for i, _, k in ZF]


def toques_forcado(t):
    (i1, y1, _), (i2, y2, _), (i3, y3, _) = t
    x, med, a, b = forcado(IBF, (i1, y1), (i2, y2), (i3, y3))
    fora = x > i3
    h, l = IBF["High"].to_numpy()[x], IBF["Low"].to_numpy()[x]
    n = sum(int(((np.abs(h / ln - 1) < 0.006) | (np.abs(l / ln - 1) < 0.006))[fora].sum()) for ln in (med, a, b))
    return n, x, med, a, b


ESC = max((PV[a:a + 3] for a in range(len(PV) - 2) if len(IBF) - PV[a + 2][0] >= 40), key=lambda t: toques_forcado(t)[0])
NT, XF, MED, SUP, INF = toques_forcado(ESC)
print(f"\nParte 11. Ibovespa: forcado com pivôs em " + ", ".join(f"{IBF.index[i]:%d/%m/%Y} ({k})" for i, _, k in ESC) +
      f". Depois do 3º pivô, {NT} pregões encostaram (0,6%) em uma das três linhas."
      "\n   Atenção: o notebook escolheu, entre todos os trios de pivôs, o de mais toques. É um exemplo, não uma prova.")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(range(len(IBF)), IBF["Close"], color="black", lw=0.9)
for ln in (MED, SUP, INF):
    ax.plot(XF, ln, color="gray", lw=0.9)
ax.set_ylim(IBF["Low"].min() * 0.97, IBF["High"].max() * 1.04)
ax.set_title("Ibovespa: Forcado de Andrews"); plt.show()

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `gaps` da Parte 2. Mostre quando o gap é de alta ou de baixa, como a coluna `dias` conta os pregões até o gap fechar e por que a Parte 1 ajusta os preços com `ajusta` antes da conta." Confira com a própria Parte 2: na linha "Todas (20 ações)", o livro tem 13.887 gaps, 76% fechados no pregão e 96% em até 60, e a última linha compara essa contagem com a feita sem o ajuste de proventos. Com a cópia local criada hoje, os números podem diferir um pouco dos do livro, porque o provedor corrige o histórico.
2. **Adaptar:** "Numa célula nova depois da Parte 2, rode `gaps` e `resumo` para [ATIVO]: baixe com `baixa`, limpe com `limpa` e ajuste com `ajusta`. Não mexa na lista `ACOES` nem na definição do gap." Antes de olhar o ativo novo, confira se a tabela da Parte 2 continua igual à Tabela 4.1 do livro (PETR4 com 66% dos gaps fechados no pregão e 94% em até 60). A estatística de um ativo só não vira regra: precisa do teste fora da amostra (Capítulo 27).

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque o mínimo do gap para 1% e para 5%. O que acontece com a fração que fecha no mesmo dia?
2. Na Parte 9, mude o prazo de 'olhar' de 10 para 20 pregões. Algum filtro muda de posição?
3. Na Parte 5, rode os níveis com toques para a sua ação preferida.
4. Na Parte 8, exija volume duas vezes acima da média. O dia de reversão melhora?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.